In [16]:
import os
import sys
import shutil
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import DataLoader

In [ ]:
script_name = source_path+"/scripts/torch_train_on_rep.py"
source_file_train='icdar_train_df_patches_20250716_11370'
source_file_val='icdar_train_df_patches_20250716_120511'
extra_view=True
extra_integration_mode = 'concat'  # 'concat' or 'add'
selected_FE = 'clip-vit-large-patch14'
input_dir=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction\\extracted_representation'

data_augmentation = True
suffix = '_augmented' if data_augmentation else ''
train_filename = input_dir+f'\\train{suffix}\\{selected_FE}_features_{source_file_train}.csv'
val_filename = input_dir+f'\\val\\{selected_FE}_features_{source_file_val}.csv'
extra_train_filename = input_dir+'\\extra_view'+f'\\train{suffix}\\{selected_FE}_features_{source_file_train}.csv' if extra_view else None
extra_val_filename = input_dir+'\\extra_view'+f'\\val\\{selected_FE}_features_{source_file_val}.csv' if extra_view else None

loss_criterion = 'CrossEntropyLoss'
model_name = 'MLPClassifier1'
total_epochs = 100
use_profiler = False
profiler_config = None
save_path = source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction\\torch_model_trained_on_rep\\{model_name}'
file_IO.access_or_create_dir(save_path)
checkpoint_path=save_path+'\\checkpoints'
file_IO.access_or_create_dir(checkpoint_path)
plot_every = 1
patience = 10
log_grad_norm = True
run_epochs = total_epochs
use_amp = False
val_percentage = 1.0
batch_size=64
aggregation_mode = 'mean'  # 'mean' or 'max'

In [ ]:
weight_decay = 1e-4  # Weight decay for the optimizer 
lr = 1e-3  # Learning rate for the classification head
lr_final = 1e-8  # Final learning rate for the backbone

#for full fine tuning
optimizer_phases = [total_epochs]  # Example: [1, 4, 95] for 100 epochs
optim_config = {
    'optimizer_phases':optimizer_phases,  # Example: [10, 10, 80] for 100 epochs
    'phase_layers_to_freeze':[[]],
    'phase_scheduling': ['no_scheduling'],
    'phase_optimizer':['Adam'],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
    'phase_lr': [lr],
    'phase_optimizer_hyperparams': [{}],
    'phase_scheduler_hyperparams': [{}],
}
file_IO.save_args(args,save_path)  # Save the arguments to a file

In [ ]:
args = script_launching.DotDict(
    data_augmentation=data_augmentation,
    extra_view=extra_view,
    loss_criterion=loss_criterion,
    model_name=model_name,
    total_epochs=total_epochs,
    patience=patience,
    log_grad_norm=log_grad_norm,
    use_amp = use_amp,
    batch_size=batch_size,
    val_percentage=val_percentage,
    weight_decay=weight_decay,
    lr=lr,
    lr_final=lr_final,
    optim_config=optim_config,
    aggregation_mode=aggregation_mode,
    extra_integration_mode=extra_integration_mode,
)

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Define datasets and group by page
train_df = pd.read_csv(train_filename)
val_df = pd.read_csv(val_filename)

if extra_view:
    train_df_extra = pd.read_csv(extra_train_filename)
    val_df_extra = pd.read_csv(extra_val_filename)
    train_df = dataframes.merge_dfs(train_df, train_df_extra, mode=extra_integration_mode)
    val_df = dataframes.merge_dfs(val_df, val_df_extra, mode=extra_integration_mode)

train_df = dataframes.aggregate_dfs(train_df,mode=aggregation_mode)
val_df = dataframes.aggregate_dfs(val_df,mode=aggregation_mode)
#train_df=file_IO.change_filename_from_to(train_df, fr=saved, to=running
#cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
cols_to_keep = [c for c in train_df.columns if c.startswith('f') and len(c) > 1 and c[1].isdigit()]
in_features = len(cols_to_keep)  # Number of features from the model output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device is: ",device)

train_dataset = dataframes.CustomExtractedDataset(train_df, label_column='male')
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = dataframes.CustomExtractedDataset(val_df, label_column='male')
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

print(f"[GPU Memory] Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB | Reserved: {torch.cuda.memory_reserved() / 1e6:.2f} MB")

loss_fn = training_utils.get_criterion(name=loss_criterion)
model = model_utils.get_classification_head(name=model_name, in_features=in_features, num_classes=2)

training_utils.train_fine(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    device=device,
    total_epochs=total_epochs,
    loss_fn=loss_fn,
    use_profiler=use_profiler,
    profiler_config=profiler_config,
    save_path=save_path,
    plot_every=plot_every,
    early_stopping_patience=patience,
    checkpoint_path=checkpoint_path+"\\checkpoint.pt",
    log_grad_norm=log_grad_norm,
    run_epochs=run_epochs,
    use_amp=use_amp,
    val_percentage=val_percentage,  # Use 10% of validation data for linear evaluation
    optim_config=optim_config,  # e.g., 'Adam', 'SGD', 'AdamW'
    # ... other parameters
)

best_checkpoint = os.path.join(checkpoint_path, "checkpoint_best.pt")
destination = os.path.join(save_path, "checkpoint_best.pt")
shutil.copy2(best_checkpoint, destination)

Device is:  cuda
Extracted 768 feature columns:
Extracted 768 feature columns:
[GPU Memory] Allocated: 20.00 MB | Reserved: 27.26 MB
Model size: 0.59 MB
🔄 Resuming from checkpoint: c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\clip-vit-large-patch14\representation_extraction\torch_model_trained_on_rep\MLPClassifier1\checkpoints\checkpoint.pt
Setting optimizer for phase 0: Adam with learning rate 0.001
Freezing [] parameters
Trainable parameters after freezing: 148,034

Epoch 24 - Optimization Phase: 0


Epoch 25/100 [Val]: 100%|██████████| 89/89 [00:00<00:00, 261.95it/s]


Epoch 25| Train Accuracy 0.7189| Train Loss: 0.5392 | Val Acc: 0.6657 | Val Loss: 0.6146 | LR: 0.001000 | Avg Grad Norm: 0.5907 | Epoch Time: 7.74s | Val Time: 0.35s
⏳ No improvement for 1 epoch(s)

Epoch 25 - Optimization Phase: 0


Epoch 26/100 [Val]: 100%|██████████| 89/89 [00:00<00:00, 266.35it/s]


Epoch 26| Train Accuracy 0.7224| Train Loss: 0.5356 | Val Acc: 0.6736 | Val Loss: 0.6223 | LR: 0.001000 | Avg Grad Norm: 0.6032 | Epoch Time: 7.72s | Val Time: 0.34s
⏳ No improvement for 2 epoch(s)

Epoch 26 - Optimization Phase: 0


...

Epoch 29/100 [Val]: 100%|██████████| 89/89 [00:00<00:00, 264.66it/s]


Epoch 29| Train Accuracy 0.7243| Train Loss: 0.5301 | Val Acc: 0.6618 | Val Loss: 0.6272 | LR: 0.001000 | Avg Grad Norm: 0.6295 | Epoch Time: 7.10s | Val Time: 0.34s
⏳ No improvement for 5 epoch(s)

Epoch 29 - Optimization Phase: 0


Epoch 30/100 [Train]:  35%|███▌      | 372/1058 [00:02<00:04, 153.76it/s]

KeyboardInterrupt: 

Epoch 30/100 [Train]:  35%|███▌      | 372/1058 [00:18<00:04, 153.76it/s]

# reload

In [20]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import utils.script_launching as script_launching
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(script_launching)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching = reload_modules()